# Saadi quickstart

Index a long document once, then answer questions about it many times. This notebook runs the whole path on a hosted model: it indexes a synthetic report, shows the entity memory and the summary hierarchy the model built, retrieves and answers, appends text without rebuilding, and loads the saved state again.

You need an OpenAI API key (about ten cents of gpt-4.1-mini for the whole notebook). Any OpenAI-compatible provider works too; see the comment in the key cell.


In [ ]:
%pip install -q 'saadi[openai]'

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
# For Groq, Together, or another OpenAI-compatible server, also set:
# os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'

In [ ]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/ZeinabTaghavi/SAADI/main/examples/aurora_report/document.txt", "aurora_report.txt"
)
print(open("aurora_report.txt").read()[:600], "...")

## Index once

One model call per chunk, plus one per summary fold. Small chunks are used here so the report folds into a hierarchy; the paper's default is 500 tokens.


In [ ]:
from saadi import Saadi

doc = Saadi(llm="openai:gpt-4.1-mini", state_dir="saadi_state/aurora", chunk_size=100)
report = doc.index("aurora_report.txt")
report

## What the model built

`memory` holds one record per entity with aliases and accumulated descriptions. `history` holds the chunks (level 0) and the fold summaries above them.


In [ ]:
for entity in doc.memory:
    print(f"{entity.name} {list(entity.aliases)}")
    for description in entity.descriptions:
        print("   -", description)

In [ ]:
for item in doc.history.get(1, []):
    print(f"L1 fold over chunks {item.start_step}-{item.end_step}:")
    print("  ", item.text, "\n")

## Retrieve and answer

Retrieval scores the question against the chunk text, the entity memory, and the summaries, and fuses the three; no model call. `answer` adds the folded context and one generation call.


In [ ]:
question = "Who keeps the Corvin station?"
for hit in doc.retrieve(question, k=3):
    print(hit.rank, f"chunk {hit.chunk_index}", round(hit.score, 3), hit.text[:110], "...")

In [ ]:
reply = doc.answer(question)
print(reply.text)

## Extend without rebuilding

Appending text reuses every unchanged chunk from the cache; only the new tail and the fold that contains it cost model calls. Compare `model_calls` with the first report.


In [ ]:
more = (
    "In November Lake Dorn froze two weeks early. Priya Nair extended her inflow model to it over the winter, "
    "and Tomas Egger scouted a path to the new station that stays safe in snow."
)
update = doc.append(more)
print(update.chunks, "chunks,", update.new_chunks, "new,", update.model_calls, "model calls")
print(doc.answer("What did Priya Nair do over the winter?").text)

## Load it again later

The state is a plain directory. Loading it needs no model for retrieval; pass `llm=` to answer or append.


In [ ]:
again = Saadi.load("saadi_state/aurora")
print(again)
print(again.retrieve("Who leads the Aurora team?", k=1)[0].text[:120], "...")

In [ ]:
import os

for name in sorted(os.listdir("saadi_state/aurora")):
    print(name)

## Your own documents

Upload a `.txt`, `.md`, or a conversation `.csv` with `role` and `text` columns through the Files panel on the left, then:

```python
doc = Saadi(llm='openai:gpt-4.1-mini', state_dir='saadi_state/mine')
doc.index('my_document.txt')
doc.answer('...')
```

Conversation transcripts work best with `prompts='conversation'`. Everything else, other providers, your own retriever or prompts, is in the [documentation](https://github.com/ZeinabTaghavi/SAADI/tree/main/docs).
